# 📘 Chapter 03. Shallow CNN 모델 및 학습 Loop 구현

> **학습 목표**
> - PyTorch를 이용하여 간단한 CNN 모델을 **직접 구현**할 수 있다.
> - MNIST 데이터셋을 로드하고 전처리하는 과정을 이해한다.
> - Training Loop의 각 단계(Forward → Loss → Backward → Update)를 체험한다.
> - 학습 결과를 시각화하고 모델 성능을 평가할 수 있다.

---

### 📍 오늘 만들 모델 구조

```
입력 (28×28×1)  MNIST 손글씨 이미지
      │
      ▼
  Conv1 (3×3, 16필터, padding=1) + ReLU + MaxPool(2×2)
      │   → 14×14×16
      ▼
  Conv2 (3×3, 32필터, padding=1) + ReLU + MaxPool(2×2)
      │   → 7×7×32
      ▼
  Flatten → 7×7×32 = 1568
      │
      ▼
  Fully Connected (1568 → 10)
      │
      ▼
  출력: 0~9 숫자 분류
```

---
## 1. 환경 설정 및 라이브러리 임포트

PyTorch와 필요한 라이브러리들을 불러옵니다.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

# GPU 사용 가능 여부 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')

---
## 2. MNIST 데이터 로딩 및 탐색

### 📌 MNIST 데이터셋이란?
- 0~9 손글씨 숫자 이미지 70,000장 (학습 60,000 + 테스트 10,000)
- 이미지 크기: **28 × 28 × 1** (흑백)
- 픽셀 값: 0~255 (정규화 후 0~1)

### 📌 transforms.ToTensor()가 하는 일
1. PIL Image(또는 NumPy 배열)를 PyTorch **Tensor**로 변환
2. 픽셀 값의 범위를 **0~255 → 0.0~1.0** 으로 자동 정규화
3. 차원 순서를 **(H, W, C) → (C, H, W)** 로 변경 (PyTorch 표준)

In [ ]:
# 데이터 전처리: 이미지를 Tensor로 변환 (0~255 → 0~1 정규화 포함)
transform = transforms.ToTensor()

# MNIST 학습 데이터 다운로드
train_dataset = torchvision.datasets.MNIST(
    root='./data',          # 데이터 저장 경로
    train=True,             # 학습용 데이터
    download=True,          # 없으면 자동 다운로드
    transform=transform     # 전처리 적용
)

# MNIST 테스트 데이터 다운로드
test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,            # 테스트용 데이터
    download=True,
    transform=transform
)

print(f'학습 데이터 수: {len(train_dataset):,}장')
print(f'테스트 데이터 수: {len(test_dataset):,}장')

### 🔍 데이터 구조 확인

실제 데이터가 어떤 형태(Shape)인지, 어떤 값을 가지는지 직접 확인해봅시다.

In [ ]:
# 첫 번째 샘플 가져오기
sample_image, sample_label = train_dataset[0]

print(f'이미지 타입: {type(sample_image)}')
print(f'이미지 Shape: {sample_image.shape}')   # (C, H, W) = (1, 28, 28)
print(f'이미지 dtype: {sample_image.dtype}')
print(f'픽셀 값 범위: {sample_image.min():.2f} ~ {sample_image.max():.2f}')
print(f'레이블(정답): {sample_label}')

### 🖼️ 샘플 이미지 시각화

학습 데이터의 처음 16장을 시각적으로 확인합니다.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle('MNIST 학습 데이터 샘플', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    image, label = train_dataset[i]
    ax.imshow(image.squeeze(), cmap='gray')  # (1,28,28) → (28,28)
    ax.set_title(f'Label: {label}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 3. DataLoader 구성

### 📌 DataLoader란?

모델 학습 시 데이터를 하나씩 넣는 것이 아니라, **여러 장을 묶어서(Batch)** 한꺼번에 넣어줍니다.

```
전체 학습 데이터: 60,000장
Batch Size: 64

→ 1 Batch = 64장의 이미지를 한 번에 모델에 입력
→ 1 Epoch = 전체 데이터를 1번 다 보는 것 = 60,000 / 64 ≈ 938 Batch (= 938 Step)
```

| 용어 | 의미 |
|------|------|
| **Batch Size** | 한 번에 모델에 넣는 데이터 수 |
| **Step (Iteration)** | 1 Batch를 처리하는 단위 |
| **Epoch** | 전체 데이터를 1바퀴 도는 단위 |

In [ ]:
# 하이퍼파라미터 설정
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 5

# DataLoader 생성
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True            # 매 Epoch마다 데이터 순서를 섞음 (과적합 방지)
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False           # 테스트 시에는 순서를 섞지 않음
)

# DataLoader에서 1 Batch 꺼내서 Shape 확인
sample_batch_images, sample_batch_labels = next(iter(train_loader))
print(f'1 Batch 이미지 Shape: {sample_batch_images.shape}')   # (64, 1, 28, 28)
print(f'1 Batch 레이블 Shape: {sample_batch_labels.shape}')   # (64,)
print(f'총 Batch 수 (1 Epoch): {len(train_loader)}')

---
## 4. ⭐ Shallow CNN 모델 정의 (핵심 실습)

Chapter 01에서 배운 CNN의 3가지 핵심 연산을 직접 코드로 구현합니다.

```
CNN의 기본 블록 = Convolution → Activation(ReLU) → Pooling
```

### 📌 PyTorch에서의 CNN 레이어

| 이론 | PyTorch 코드 | 설명 |
|------|-------------|------|
| Convolution | `nn.Conv2d(in_channels, out_channels, kernel_size, padding)` | 합성곱 레이어 |
| Activation | `nn.ReLU()` | 활성화 함수 |
| Pooling | `nn.MaxPool2d(kernel_size)` | 최대 풀링 |
| Flatten | `nn.Flatten()` | 2D → 1D 벡터 변환 |
| Fully Connected | `nn.Linear(in_features, out_features)` | 완전연결 레이어 |

### 📌 nn.Conv2d 파라미터 해설

```python
nn.Conv2d(
    in_channels=1,     # 입력 채널 수 (흑백=1, RGB=3)
    out_channels=16,   # 출력 채널 수 = 필터 개수 (16종류의 특징 추출)
    kernel_size=3,     # 필터 크기 (3×3)
    padding=1          # 입력 주변에 0을 1칸 채워 출력 크기를 입력과 동일하게 유지
)
```

### 📝 실습 안내
아래 코드의 **주석(`#`)을 해제**하여 모델을 완성하세요!

In [ ]:
class ShallowCNN(nn.Module):
    def __init__(self):
        super(ShallowCNN, self).__init__()

        # ============================================================
        # [실습] 아래 주석을 해제하여 CNN 레이어를 정의하세요.
        # ============================================================

        # --- Block 1: Conv → ReLU → MaxPool ---
        # 입력: (Batch, 1, 28, 28)  →  출력: (Batch, 16, 14, 14)
        # self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        # self.relu1 = nn.ReLU()
        # self.pool1 = nn.MaxPool2d(kernel_size=2)

        # --- Block 2: Conv → ReLU → MaxPool ---
        # 입력: (Batch, 16, 14, 14)  →  출력: (Batch, 32, 7, 7)
        # self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        # self.relu2 = nn.ReLU()
        # self.pool2 = nn.MaxPool2d(kernel_size=2)

        # --- Classifier ---
        # Flatten: (Batch, 32, 7, 7) → (Batch, 32*7*7) = (Batch, 1568)
        # self.flatten = nn.Flatten()
        # self.fc = nn.Linear(in_features=32 * 7 * 7, out_features=10)  # 10개 클래스 (0~9)

    def forward(self, x):
        # ============================================================
        # [실습] 아래 주석을 해제하여 Forward Pass를 완성하세요.
        #        데이터가 각 레이어를 순서대로 통과하는 과정입니다.
        # ============================================================

        # --- Block 1 ---
        # x = self.conv1(x)     # Convolution: 특징 추출
        # x = self.relu1(x)     # Activation:  비선형성 부여
        # x = self.pool1(x)     # Pooling:     공간 압축 (28→14)

        # --- Block 2 ---
        # x = self.conv2(x)     # Convolution: 더 복잡한 특징 추출
        # x = self.relu2(x)     # Activation:  비선형성 부여
        # x = self.pool2(x)     # Pooling:     공간 압축 (14→7)

        # --- Classifier ---
        # x = self.flatten(x)   # Flatten: 2D 특징 맵 → 1D 벡터
        # x = self.fc(x)        # Fully Connected: 클래스별 점수 출력

        return x

### 🔍 모델 구조 확인

모델을 생성하고, 구조와 파라미터 수를 확인합니다.

In [ ]:
# 모델 생성 및 디바이스(GPU/CPU)로 이동
model = ShallowCNN().to(device)

# 모델 구조 출력
print(model)
print('---')

# 총 파라미터 수 계산
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'총 파라미터 수: {total_params:,}')
print(f'학습 가능 파라미터 수: {trainable_params:,}')

### 🧪 더미 입력으로 Forward Pass 테스트

모델이 올바르게 정의되었는지 확인하기 위해, 랜덤 데이터를 넣어봅니다.

In [ ]:
# 더미 입력: Batch 1, Channel 1, Height 28, Width 28
dummy_input = torch.randn(1, 1, 28, 28).to(device)
dummy_output = model(dummy_input)

print(f'입력 Shape:  {dummy_input.shape}')    # (1, 1, 28, 28)
print(f'출력 Shape:  {dummy_output.shape}')   # (1, 10)  ← 10개 클래스에 대한 점수
print(f'출력 값 (raw scores): {dummy_output.detach().cpu().numpy().round(3)}')

---
## 5. 손실 함수(Loss)와 옵티마이저(Optimizer) 설정

### 📌 Loss Function: CrossEntropyLoss
- Chapter 02에서 배운 **Categorical Cross-Entropy Loss**입니다.
- PyTorch의 `nn.CrossEntropyLoss()`는 내부적으로 **Softmax + CE Loss**를 함께 처리합니다.
- 따라서 모델의 마지막 출력에 별도로 Softmax를 적용하지 않아도 됩니다.

### 📌 Optimizer: Adam
- 모델의 가중치(Weight)를 업데이트하는 알고리즘입니다.
- Loss를 줄이는 방향으로 가중치를 조금씩 조정합니다.
- Adam은 학습률(Learning Rate)을 자동 조절하는 기능이 있어 널리 사용됩니다.

```
가중치 업데이트 과정:

  1. Forward Pass  → 예측값 계산
  2. Loss 계산     → 정답과의 차이 측정
  3. Backward Pass → 각 가중치가 Loss에 기여한 정도(기울기, Gradient) 계산
  4. Optimizer Step → 기울기 방향으로 가중치를 조금 업데이트
```

In [ ]:
# 손실 함수: CrossEntropyLoss (다중 클래스 분류)
criterion = nn.CrossEntropyLoss()

# 옵티마이저: Adam
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f'Loss Function: {criterion}')
print(f'Optimizer: Adam (lr={LEARNING_RATE})')

---
## 6. ⭐ Training Loop 구현 (핵심 실습)

드디어 모델을 학습시키는 핵심 코드입니다!

### 📌 Training Loop 한 Step의 흐름

```
┌──────────────────────────────────────────────────────────┐
│  for images, labels in train_loader:  ← 1 Batch씩 반복  │
│                                                          │
│    ① Forward Pass:    outputs = model(images)            │
│       → 모델에 이미지를 넣어 예측값(10개 클래스 점수) 얻기  │
│                                                          │
│    ② Loss 계산:       loss = criterion(outputs, labels)  │
│       → 예측값과 정답의 차이(Cross-Entropy Loss) 계산      │
│                                                          │
│    ③ Backward Pass:   loss.backward()                    │
│       → 각 가중치의 기울기(Gradient) 자동 계산             │
│                                                          │
│    ④ 가중치 업데이트:  optimizer.step()                   │
│       → 기울기 방향으로 가중치를 조금씩 수정               │
│                                                          │
│    ⑤ 기울기 초기화:   optimizer.zero_grad()               │
│       → 다음 Step을 위해 기울기 누적 방지 (0으로 리셋)     │
│                                                          │
└──────────────────────────────────────────────────────────┘
```

### 📝 실습 안내
아래 코드에서 **`# [실습]`** 표시된 주석을 해제하여 Training Loop를 완성하세요!

In [ ]:
# 학습 기록 저장용 리스트
train_losses = []
train_accuracies = []
test_accuracies = []

# ============================================================
# Training Loop
# ============================================================
for epoch in range(NUM_EPOCHS):

    # --- 학습 모드 ---
    model.train()  # 모델을 학습 모드로 전환 (Dropout, BatchNorm 등이 학습 모드로 동작)
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        # 데이터를 디바이스(GPU/CPU)로 이동
        images = images.to(device)
        labels = labels.to(device)

        # ============================================================
        # [실습] 아래 주석을 해제하여 학습 Step을 완성하세요.
        # ============================================================

        # ① Forward Pass: 모델에 이미지를 넣어 예측값 계산
        # outputs = model(images)

        # ② Loss 계산: 예측값과 정답의 차이 측정
        # loss = criterion(outputs, labels)

        # ③ Backward Pass: 기울기(Gradient) 계산
        # loss.backward()

        # ④ 가중치 업데이트: 기울기 방향으로 가중치 수정
        # optimizer.step()

        # ⑤ 기울기 초기화: 다음 Step을 위해 기울기를 0으로 리셋
        # optimizer.zero_grad()

        # --- 학습 통계 기록 ---
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)  # 가장 높은 점수의 클래스 = 예측 결과
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # 100 Step마다 진행 상황 출력
        if (batch_idx + 1) % 100 == 0:
            print(f'  Epoch [{epoch+1}/{NUM_EPOCHS}], '
                  f'Step [{batch_idx+1}/{len(train_loader)}], '
                  f'Loss: {loss.item():.4f}')

    # Epoch별 학습 Loss/Accuracy 기록
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)

    # --- 평가 모드 (테스트 데이터로 성능 확인) ---
    model.eval()  # 모델을 평가 모드로 전환
    test_correct = 0
    test_total = 0

    with torch.no_grad():  # 평가 시에는 기울기 계산 불필요 (메모리 절약)
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    test_acc = 100 * test_correct / test_total
    test_accuracies.append(test_acc)

    print(f'\n📊 Epoch [{epoch+1}/{NUM_EPOCHS}] 결과')
    print(f'   Train Loss: {epoch_loss:.4f} | Train Accuracy: {epoch_acc:.2f}%')
    print(f'   Test Accuracy: {test_acc:.2f}%')
    print('-' * 60)

print('\n✅ 학습 완료!')

---
## 7. 학습 결과 시각화

학습이 잘 되었는지, Loss와 Accuracy의 변화를 그래프로 확인합니다.

### 📌 잘 학습된 모델의 그래프 특징
- **Loss**: Epoch가 진행될수록 **지속적으로 감소**
- **Accuracy**: Epoch가 진행될수록 **지속적으로 증가**
- **Train vs Test 차이가 크지 않음**: 차이가 크면 과적합(Overfitting) 의심

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

# Loss 그래프
ax1.plot(epochs_range, train_losses, 'o-', color='#e74c3c', linewidth=2, markersize=8, label='Train Loss')
ax1.set_title('Training Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy 그래프
ax2.plot(epochs_range, train_accuracies, 'o-', color='#3498db', linewidth=2, markersize=8, label='Train Accuracy')
ax2.plot(epochs_range, test_accuracies, 's--', color='#2ecc71', linewidth=2, markersize=8, label='Test Accuracy')
ax2.set_title('Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. 테스트 데이터로 예측 결과 확인

모델이 실제로 어떻게 예측하는지, 테스트 데이터 몇 장을 뽑아 시각적으로 확인합니다.

In [ ]:
# 테스트 데이터에서 16장 뽑아 예측 결과 시각화
model.eval()
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('모델 예측 결과 (테스트 데이터)', fontsize=14, fontweight='bold')

with torch.no_grad():
    for i, ax in enumerate(axes.flatten()):
        image, label = test_dataset[i]
        output = model(image.unsqueeze(0).to(device))   # (1,1,28,28) 형태로 입력
        prob = torch.softmax(output, dim=1)             # 확률로 변환
        pred = output.argmax(dim=1).item()              # 가장 높은 점수의 클래스
        confidence = prob[0][pred].item() * 100         # 확신도 (%)

        ax.imshow(image.squeeze(), cmap='gray')

        # 정답이면 초록색, 오답이면 빨간색 제목
        color = 'green' if pred == label else 'red'
        ax.set_title(f'예측:{pred} ({confidence:.0f}%)\n정답:{label}',
                     fontsize=9, color=color, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.show()

---
## 9. 모델 저장

학습이 완료된 모델의 가중치를 파일로 저장합니다.
다음 Chapter에서 이 모델을 불러와 직접 손글씨를 추론하는 데 사용합니다.

In [ ]:
# 모델 가중치 저장
save_path = 'mnist_shallow_cnn.pth'
torch.save(model.state_dict(), save_path)
print(f'✅ 모델 저장 완료: {save_path}')

---
## 📝 Chapter 03 핵심 정리

| 단계 | 코드 | 설명 |
|------|------|------|
| **데이터 로드** | `torchvision.datasets.MNIST()` | MNIST 데이터셋 다운로드 및 전처리 |
| **DataLoader** | `DataLoader(dataset, batch_size, shuffle)` | 데이터를 Batch 단위로 묶어서 제공 |
| **모델 정의** | `nn.Module` 상속, `__init__` + `forward` | Conv→ReLU→Pool 블록 2개 + FC |
| **Loss 함수** | `nn.CrossEntropyLoss()` | 다중 클래스 분류용 손실 함수 |
| **Optimizer** | `optim.Adam(model.parameters(), lr)` | 가중치 업데이트 알고리즘 |
| **학습 Loop** | `model(x)` → `loss` → `.backward()` → `.step()` | Forward → Loss → Backward → Update |
| **평가** | `model.eval()` + `torch.no_grad()` | 기울기 계산 없이 성능 측정 |
| **모델 저장** | `torch.save(model.state_dict(), path)` | 학습된 가중치를 파일로 저장 |

### 🚀 다음 Chapter에서는?
- 저장된 모델을 불러와, 직접 손으로 쓴 숫자를 인식시켜 봅니다!
- 추론(Inference) 과정과 학습의 차이를 이해합니다.